# Wind Generation Reliability Analysis — GB

Uses historical BMRS FUELHH actuals (January 2025 onwards) to assess how reliably
GB wind generation can be counted on, and what backup reserve a grid operator should
size for worst-case wind conditions.

## Research Questions
1. What is the *firm* (reliable) wind generation capacity?
2. How does reliability vary by time of day and season?
3. What is the longest observed low-wind period?
4. What reserve margin should the grid operator maintain?

In [ ]:
import sqlite3
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

warnings.filterwarnings('ignore')
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('ggplot')

pd.set_option('display.float_format', '{:.1f}'.format)
print('Setup complete')

## 1. Load Actuals from SQLite

In [ ]:
db_path = None
for candidate in [
    '../backend/data/wind_monitor.db',
    '../../backend/data/wind_monitor.db',
    os.path.join(os.path.dirname(os.getcwd()), 'backend', 'data', 'wind_monitor.db'),
]:
    if os.path.exists(candidate):
        db_path = candidate
        break

if db_path is None:
    raise FileNotFoundError(
        'Database not found. Fetch actuals first:\n'
        '  curl "http://localhost:3000/api/actual-generation?start=2025-01-01&end=TODAY"'
    )

conn = sqlite3.connect(db_path)
raw = pd.read_sql_query('SELECT start_time, generation FROM actuals ORDER BY start_time', conn)
conn.close()

df = raw.copy()
df['start_time']    = pd.to_datetime(df['start_time'], utc=True)
df['uk_time']       = df['start_time'].dt.tz_convert('Europe/London')
df.rename(columns={'generation': 'generation_mw'}, inplace=True)

# Remove physically impossible readings
df = df[(df['generation_mw'] >= 0) & (df['generation_mw'] < 25000)]

print(f'Records  : {len(df):,}')
print(f'From     : {df["uk_time"].min().date()}')
print(f'To       : {df["uk_time"].max().date()}')
print(f'Days     : {(df["uk_time"].max() - df["uk_time"].min()).days}')
if len(df) == 0:
    print('No data — fetch via the API first.')

## 2. Descriptive Statistics

In [ ]:
if len(df) > 0:
    # UK installed wind capacity ~30 GW as of 2025 (onshore ~15 GW + offshore ~15 GW)
    INSTALLED_CAPACITY_MW = 30_000

    desc = df['generation_mw'].describe(
        percentiles=[0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
    )
    cap_factor = df['generation_mw'].mean() / INSTALLED_CAPACITY_MW * 100
    zero_pct   = (df['generation_mw'] < 50).mean() * 100

    print('Generation Statistics (MW)')
    print('=' * 35)
    for name, val in desc.items():
        print(f'  {name:<10}: {val:>9,.1f} MW')
    print()
    print(f'  Capacity factor     : {cap_factor:.1f}%')
    print(f'  Near-zero (<50 MW)  : {zero_pct:.1f}% of half-hours')
    print(f'  Installed capacity  : {INSTALLED_CAPACITY_MW:,} MW (assumed)')

## 3. Percentile Analysis and Distribution

Key percentiles for reliability planning:
- **P10** = exceeded 90% of the time → *reliable floor* (recommended firm capacity)
- **P50** = median (exceeded 50% of the time)
- **P90** = exceeded only 10% of the time → favourable / high-wind conditions

In [ ]:
if len(df) > 0:
    pct_levels = [5, 10, 25, 50, 75, 90, 95]
    pcts = {f'P{p}': np.percentile(df['generation_mw'], p) for p in pct_levels}

    print('Percentile Table')
    print('-' * 32)
    for name, val in pcts.items():
        bar = '#' * int(val / 400)
        print(f'  {name}: {val:>8,.0f} MW  {bar}')

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Histogram with P10/P50/P90 marked
    axes[0].hist(df['generation_mw'], bins=80, color='#2563eb', alpha=0.7, edgecolor='none')
    for p, col in [(10, '#ef4444'), (50, '#f59e0b'), (90, '#16a34a')]:
        v = pcts[f'P{p}']
        axes[0].axvline(v, color=col, ls='--', lw=2, label=f'P{p}: {v:,.0f} MW')
    axes[0].set(title='Distribution of Half-Hourly Wind Generation',
                xlabel='Generation (MW)', ylabel='Half-hour periods')
    axes[0].legend(fontsize=9)

    # Duration curve (exceedance probability)
    gen_sorted   = np.sort(df['generation_mw'].values)[::-1]
    exceedance   = np.linspace(0, 100, len(gen_sorted))
    axes[1].plot(exceedance, gen_sorted, color='#2563eb', lw=2)
    axes[1].fill_between(exceedance, gen_sorted, alpha=0.1, color='#2563eb')
    for p, col in [(10, '#ef4444'), (50, '#f59e0b'), (90, '#16a34a')]:
        axes[1].axvline(p, color=col, ls='--', lw=1.5, label=f'P{p}: {pcts[f"P{p}"]:,.0f} MW')
    axes[1].set(title='Generation Duration Curve',
                xlabel='% of time generation exceeds this level', ylabel='Generation (MW)')
    axes[1].legend(fontsize=9)

    plt.tight_layout()
    plt.savefig('../generation_distribution.png', dpi=120, bbox_inches='tight')
    plt.show()

## 4. Temporal Reliability Patterns

Does wind availability vary by time of day or season?
This is critical for scheduling — predictable low-wind windows require planned backup.

In [ ]:
if len(df) > 0:
    df['hour']  = df['uk_time'].dt.hour
    df['month'] = df['uk_time'].dt.month
    df['dow']   = df['uk_time'].dt.dayofweek

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # By hour of day — show median + P10-P90 band
    hour_stats = df.groupby('hour')['generation_mw'].agg(
        median = 'median',
        p10    = lambda x: np.percentile(x, 10),
        p90    = lambda x: np.percentile(x, 90),
    )
    h = hour_stats.index
    axes[0].fill_between(h, hour_stats['p10'], hour_stats['p90'],
                         alpha=0.18, color='#2563eb', label='P10-P90 range')
    axes[0].plot(h, hour_stats['median'], color='#2563eb', lw=2,   label='Median (P50)')
    axes[0].plot(h, hour_stats['p10'],    color='#ef4444', lw=1.5,
                 ls='--', label='P10 (reliable floor)')
    axes[0].set(title='Generation by Hour of Day (UK)', xlabel='Hour', ylabel='MW')
    axes[0].legend(fontsize=8)

    # By month
    mth_map = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
    mth_stats = df.groupby('month')['generation_mw'].agg(
        median = 'median',
        p10    = lambda x: np.percentile(x, 10),
        p90    = lambda x: np.percentile(x, 90),
    )
    x = range(len(mth_stats))
    axes[1].fill_between(x, mth_stats['p10'], mth_stats['p90'],
                         alpha=0.18, color='#2563eb')
    axes[1].plot(x, mth_stats['median'], color='#2563eb', lw=2, label='Median')
    axes[1].plot(x, mth_stats['p10'],    color='#ef4444', lw=1.5, ls='--', label='P10')
    axes[1].set_xticks(list(x))
    axes[1].set_xticklabels([mth_map[m] for m in mth_stats.index])
    axes[1].set(title='Generation by Month', ylabel='MW')
    axes[1].legend(fontsize=8)

    plt.suptitle('Temporal Reliability Patterns', y=1.01, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../temporal_reliability.png', dpi=120, bbox_inches='tight')
    plt.show()

## 5. Consecutive Low-Wind Periods

A key grid planning input: *how long* can wind stay persistently low?
This determines the required **duration** (not just capacity) of backup generation.

Definition: a low-wind period is a continuous run of half-hours at or below the P10 level.

In [ ]:
if len(df) > 0:
    p10_thresh = pcts['P10']

    df_s = df.sort_values('start_time').reset_index(drop=True)
    df_s['is_low'] = df_s['generation_mw'] <= p10_thresh

    # Identify run boundaries (consecutive runs of True)
    df_s['run_id'] = (df_s['is_low'] != df_s['is_low'].shift()).cumsum()

    low_runs = (
        df_s[df_s['is_low']]
        .groupby('run_id')
        .agg(
            start       = ('start_time', 'min'),
            end         = ('start_time', 'max'),
            n_periods   = ('is_low', 'count'),
            min_gen     = ('generation_mw', 'min'),
            mean_gen    = ('generation_mw', 'mean'),
        )
        .reset_index(drop=True)
    )
    low_runs['duration_h'] = low_runs['n_periods'] * 0.5

    print(f'P10 threshold : {p10_thresh:,.0f} MW')
    print(f'Low-wind runs : {len(low_runs):,}')
    print(f'Longest run   : {low_runs["duration_h"].max():.1f} hours')
    print(f'Mean run      : {low_runs["duration_h"].mean():.1f} hours')
    print()
    print('Top 5 longest low-wind periods:')
    top5 = (
        low_runs.nlargest(5, 'duration_h')
        [['start', 'duration_h', 'min_gen']]
        .rename(columns={'start': 'Start (UTC)', 'duration_h': 'Hours', 'min_gen': 'Min MW'})
    )
    print(top5.to_string(index=False))

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(low_runs['duration_h'], bins=30, color='#ef4444', alpha=0.75, edgecolor='none')
    ax.set(
        title=f'Duration of Low-Wind Events (generation <= P10 = {p10_thresh:,.0f} MW)',
        xlabel='Duration (hours)',
        ylabel='Number of events',
    )
    plt.tight_layout()
    plt.savefig('../low_wind_durations.png', dpi=120, bbox_inches='tight')
    plt.show()

## 6. Reliability Recommendation

### Chosen Approach: P10 Percentile Method

We recommend the **P10 generation level** as the *firm (reliable) capacity* that grid
operators can count on from GB wind.

**Justification:**
- P10 = the generation level **exceeded 90% of all half-hours** — a robust, well-understood threshold
- It accommodates the worst ~10% of conditions (storms, maintenance, prolonged low-pressure)
- Distribution-free: no normality assumption required (wind is right-skewed, making Gaussian CI unreliable)
- Industry standard: IEC 61400-12 and most wind resource assessments use P90 exceedance (= P10 here)
- Conservative but not extreme: P5 would be overly cautious; P25 too optimistic

**Alternatives considered and rejected:**
1. *Mean − 2σ* (95% CI lower bound): assumes normal distribution, but wind is right-skewed — this underestimates tail risk
2. *Worst observed month*: too sensitive to the limited data window and outlier events
3. *P5*: overly conservative for planning; would leave too much backup idle most of the time

In [ ]:
if len(df) > 0:
    mean_gen   = df['generation_mw'].mean()
    std_gen    = df['generation_mw'].std()
    p10_firm   = pcts['P10']
    p5_firm    = pcts['P5']
    gauss_lb   = max(0.0, mean_gen - 2 * std_gen)
    backup_req = mean_gen - p10_firm

    print('Reliability Recommendation')
    print('=' * 48)
    print(f'  P10 firm capacity  : {p10_firm:>9,.0f} MW  <- RECOMMENDED')
    print(f'  P5  (conservative) : {p5_firm:>9,.0f} MW')
    print(f'  Mean - 2sigma      : {gauss_lb:>9,.0f} MW  (assumes normal distribution)')
    print()
    print(f'  Mean generation    : {mean_gen:>9,.0f} MW')
    print(f'  Std deviation      : {std_gen:>9,.0f} MW')
    print()
    print(f'  -> Wind can reliably deliver at least {p10_firm:,.0f} MW  90% of the time.')
    print(f'  -> Backup reserve needed above firm: ~{backup_req:,.0f} MW (mean - P10)')
    print(f'  -> Longest observed low-wind period: {low_runs["duration_h"].max():.0f} hours')
    print(f'     (backup must sustain this duration, not just peak capacity)')

    # Visualise
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.hist(df['generation_mw'], bins=100, color='#2563eb', alpha=0.55,
            edgecolor='none', label='Observed generation')
    ax.axvspan(0, p10_firm, alpha=0.12, color='#ef4444')
    ax.axvline(p10_firm, color='#ef4444', lw=2.5,
               label=f'P10 firm capacity: {p10_firm:,.0f} MW')
    ax.axvline(mean_gen, color='#f59e0b', lw=2, ls='--',
               label=f'Mean: {mean_gen:,.0f} MW')
    ax.set(
        title='Reliability Recommendation — P10 as Firm Wind Capacity',
        xlabel='Generation (MW)',
        ylabel='Half-hour periods',
    )
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig('../reliability_recommendation.png', dpi=120, bbox_inches='tight')
    plt.show()

## 7. Summary & Limitations

### Key Findings *(update after running with real data)*

| Metric | Value |
|---|---|
| P10 firm capacity | __,___ MW |
| Capacity factor | __ % |
| Median generation | __,___ MW |
| Longest low-wind event | __ hours |
| Required backup vs mean | __,___ MW |

### Important Limitations

1. **Limited data window**: only ~2-3 months available from Jan 2025. UK wind output
   typically peaks in winter — analysis may **over-estimate** P10 if summer data
   (June–August, generally lower wind) is not yet included.

2. **No seasonal cycle**: reliable P10 estimation ideally uses 12+ months to capture
   full seasonality. Current estimate should be treated as provisional.

3. **Aggregated data**: FUELHH combines onshore and offshore. Offshore tends to be
   more consistent; the aggregate hides geographic diversity effects.

4. **Growing installed capacity**: UK wind capacity is expanding — historical MW values
   may under-represent future generation potential.

Despite these limitations, the **P10 approach** provides a conservative, industry-standard
baseline. Revisit once ≥12 months of data are available.